# Table summary


In [1]:
# general
from pathlib import Path
import pandas as pd
import geopandas as gpd
import xarray as xr
import yaml 
import os

wd = Path('/home/rooda/OneDrive/Projects/Whatif-Andes/')
os.chdir(wd)

# import config and local module
with open("config.yml") as f:
    cfg = yaml.safe_load(f)

In [2]:
# Regions and Scenarios
regions = {
    'Outer tropics': (-20, -8),
    'Desert Andes': (-30, -20),
    'Central Andes': (-35, -30),
    'North Patagonia': (-45, -35),
    'South Patagonia': (-52, -45),
    'Fuegian Andes': (-56, -52)}

## Data

In [6]:
# polygons / shapes
df = gpd.read_file("data/hydro/basins_hydroATLAS_lev08.gpkg").set_index("gauge_id")
df = df[['glacier_area_RGI60', 'basin_area', 'basin_cenlat', 'prcp_mean_ERA5']]
df = df[df.glacier_area_RGI60 > 0]
df["glacier_area_RGI60"] = df.glacier_area_RGI60 * 0.01 * df.basin_area

# droughts dataset
droughts = pd.read_csv("data/hydro/droughts_hydroatlas_whatif.csv")
droughts = droughts.pivot(index="gauge_id", columns="scenario")
droughts.columns = ['_'.join(col).strip() for col in droughts.columns.values]
droughts.index = droughts.index.astype(str)
df = df.join(droughts)

# total runoff in mm/day to m3/s
hydroatlas_runoff_sim = xr.open_dataset("data/hydro/runoff_sim_hydroatlas_hist.nc").q_mm_sim
hydroatlas_runoff_sim = hydroatlas_runoff_sim.reindex(gauge_id=df.index, fill_value=0)
hydroatlas_runoff_sim = hydroatlas_runoff_sim*(df.basin_area.values*1e6)[:, None] / (1000*86400)
hydroatlas_runoff_sim = hydroatlas_runoff_sim.resample(date="YS").mean().mean(dim = "date")
df = df.join(hydroatlas_runoff_sim.rename("total_runoff").to_pandas())

# glacier runoff in m3/s
hydroatlas_gmelt = xr.open_dataset("data/glacier/run_output_historical.nc").glacier_melt
hydroatlas_gmelt = hydroatlas_gmelt.resample(date="YS").mean().mean(dim = "date")
df = df.join(hydroatlas_gmelt.rename("glacier_runoff").to_pandas())

# glacier evolution
df_glaciers = pd.read_csv(Path(cfg["dir"]["pipeline"]) / "oggm15" / "glacier_statistics.csv", index_col=0, low_memory=False)
df_glaciers["region_name"] = df_glaciers["cenlat"].apply(
    lambda lat: next((r for r, (lo, hi) in regions.items() if lo <= lat < hi), "Unassigned"))

files = [f for f in Path(cfg["dir"]["pipeline"]).glob("oggm15/*output*") if "historical" not in f.name]
warming_levels = [f.name.split('_')[2] for f in files]

def get_regional_volume(ds, time_sel):
    vol = ds.volume.sel(time=time_sel)
    return vol.assign_coords(
        region_name=("rgi_id", df_glaciers["region_name"].reindex(vol.rgi_id.values).values)
    ).groupby("region_name").sum("rgi_id")

df_historical = get_regional_volume(xr.open_dataset(Path(cfg["dir"]["pipeline"]) / "oggm15" / "run_output_historical.nc"), 2020)

df_future = xr.concat([get_regional_volume(xr.open_dataset(f), 2099) for f in files], dim="warming_level")
df_future = df_future.assign_coords(warming_level=warming_levels).groupby("warming_level").mean()

glacier_loss = (100 * (1 - df_future / df_historical)).drop_vars(["hydro_month", "calendar_month"]).to_pandas().reset_index()
glacier_loss["warming_level"] = df_future.warming_level.values
glacier_loss = glacier_loss.set_index("warming_level")
glacier_loss.index = "glacier_loss_" + glacier_loss.index

## Table

In [7]:
# regional values from runoff simulation
df_agg = pd.DataFrame(columns=regions.keys())
sum_cols = ['glacier_area_RGI60', 'basin_area', 'total_runoff', 'glacier_runoff']
for region, (lat_min, lat_max) in regions.items():
    mask = (df['basin_cenlat'] >= lat_min) & (df['basin_cenlat'] < lat_max)
    mean_cols = df.columns.difference(sum_cols)
    weights = df.loc[mask, 'basin_area']
    weighted_means = df.loc[mask, mean_cols].apply(lambda x: (x * weights).sum() / weights[x.notna()].sum())
    df_agg[region] = pd.concat([
        df.loc[mask, sum_cols].sum(),
        weighted_means
    ])

df_agg.loc["glacier_area_RGI60"] = df_agg.loc["glacier_area_RGI60"] * 100 / df_agg.loc["basin_area"]
df_agg.loc["glacier_runoff_rel"] =  df_agg.loc["glacier_runoff"] * 100 / df_agg.loc["total_runoff"]
df_agg = pd.concat([df_agg, glacier_loss], axis=0)

In [8]:
df_agg.round(2).to_csv("data/summary_table.csv")